# Chương 3 — Ứng dụng 1: Phân tích giỏ hàng và sinh luật kết hợp

**Đồ án Khai phá dữ liệu, Group 23 — FP-Growth (Han, Pei, Yin, Mao 2004).**

Đề cho chọn một trong hai hướng cho Ứng dụng 1. Nhóm chọn **hướng A — phân tích
giỏ hàng (Market Basket Analysis)**: từ các tập phổ biến, sinh luật kết hợp thỏa
minsup và minconf, lấy top 10 luật theo **lift** và giải thích ý nghĩa kinh doanh.

Lý do chọn hướng A thay vì hướng log hệ thống:

1. Đề nêu rõ sản phẩm phải có (luật kết hợp, top 10 theo lift, ý nghĩa kinh
   doanh), nên không có chỗ hiểu sai khi chấm.
2. Bước biến đổi dữ liệu thành giao dịch là tự nhiên: một hoá đơn là một giao
   dịch. Hướng log còn phải tự quyết định cách chia session, dễ gây tranh luận.
3. Toàn bộ phần tính toán chỉ cần thư viện chuẩn của Python, đúng ràng buộc của
   đề (chỉ được dùng `numpy`, `pandas`, `bitarray` ngoài thư viện chuẩn).

**Mọi kết quả trong notebook này đều do cài đặt FP-Growth của nhóm sinh ra**
(`src/algorithm/fpgrowth.py`) và bộ sinh luật của nhóm (`src/rules.py`). Không
dùng `mlxtend`, `efficient-apriori`, `arules` hay bất kỳ thư viện khai thác tập
phổ biến nào.

## Nội dung

| Mục | Nội dung |
|---|---|
| 1 | Dữ liệu: CSDL Groceries, mô tả và kiểm chứng |
| 2 | Biến đổi dữ liệu thành giao dịch (`encode`) |
| 3 | Khai thác tập phổ biến bằng FP-Growth của nhóm |
| 4 | Sinh luật kết hợp: support, confidence, lift, leverage |
| 5 | **Top 10 luật theo lift và ý nghĩa kinh doanh** |
| 6 | Vì sao xếp theo lift chứ không phải confidence |
| 7 | Null model: các mẫu tìm được có thật hay chỉ do item bán chạy? |
| 8 | Ảnh hưởng của minsup và minconf |
| 9 | Chạy trên CSDL lớn (Retail) và bài học về lift ở dữ liệu thưa |
| 10 | Kết luận và đề xuất hành động |

In [ ]:
# Chuẩn bị môi trường. Ô này chạy được cả trên Google Colab lẫn trên máy cá nhân.
import os
import sys

REPO_URL = "https://github.com/KhoaTranFitus/Frequent_Itemset.git"
REPO_DIR = "Frequent_Itemset"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
else:
    # đi lên tới thư mục gốc repo (thư mục chứa src/ và data/)
    while not os.path.isdir("src") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Thư mục làm việc:", os.getcwd())
for path in ("src/algorithm/fpgrowth.py", "src/rules.py", "src/apps/basket.py",
             "data/app1/groceries.csv", "data/benchmark/retail.txt"):
    print(("  có    " if os.path.exists(path) else "  THIẾU ") + path)

In [ ]:
import time
from collections import Counter

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

from src.algorithm.fpgrowth import fpgrowth
from src.apps.basket import (describe, load_dataset, read_baskets,
                             shuffle_keep_marginals)
from src.config import (APP1_MINCONF, APP1_MINCONFS, APP1_MINSUP, APP1_MINSUPS,
                        APP1_MINSUP_RETAIL, APP1_SHUFFLES, GROCERIES, SEED,
                        to_min_count)
from src.rules import format_rule, generate_rules, rule_names, sort_rules
from src.utils import encode, invert_code

# Bảng màu và kiểu đồ thị dùng chung cho mọi hình, để các hình trong báo cáo
# trông như một bộ. Dải RAMP là MỘT màu đậm dần, dùng cho đại lượng có thứ tự
# (minsup); hai màu BLUE/ORANGE dùng khi có hai chuỗi không so được bậc.
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"
BLUE, ORANGE = "#2a78d6", "#eb6834"
RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]
SEQ = LinearSegmentedColormap.from_list("blue_seq", [
    "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 110,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK_2,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,        # lưới nằm DƯỚI dữ liệu
    "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 9,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 5,
    "figure.autolayout": True,                        # không bị cắt nhãn
})
pd.set_option("display.width", 140)

print("python", sys.version.split()[0], "| matplotlib", matplotlib.__version__,
      "| pandas", pd.__version__)
print("Ngưỡng mặc định lấy từ src/config.py:",
      f"minsup {APP1_MINSUP}, minconf {APP1_MINCONF}, seed {SEED}")

## 1. Dữ liệu: CSDL Groceries

`data/app1/groceries.csv` — dữ liệu giỏ hàng thật của một cửa hàng bán lẻ, là bộ
`Groceries` đi kèm gói R `arules` (Hahsler, Grün, Hornik 2005). Mỗi dòng là một
giỏ, các món cách nhau bằng dấu phẩy, item là **tên đọc được**:

```
citrus fruit,semi-finished bread,margarine,ready soups
tropical fruit,yogurt,coffee
whole milk
```

**Vì sao chọn bộ này chứ không chỉ dùng `retail.txt` đã có trong repo:** item của
Retail đã bị ẩn danh thành số, mà đề yêu cầu *giải thích ý nghĩa kinh doanh* của
luật — không có tên món thì không giải thích được gì. Mục 9 vẫn chạy lại toàn bộ
quy trình trên Retail để cho thấy phương pháp làm việc được ở quy mô 88.162 giỏ.

**Vì sao bộ này dùng để kiểm chứng được:** số liệu của Groceries đã được công bố
rộng rãi, nên nhóm có mốc để đối chiếu cài đặt của mình. Notebook sẽ kiểm bốn
mốc: `support(whole milk)`, giá trị của hai luật kinh điển, và số luật ở hai cặp
ngưỡng. Nhóm đối chiếu **kết quả số**, không đọc và không sao chép code của
`arules`.

In [ ]:
baskets = read_baskets(GROCERIES)
stats = describe(baskets)

print(f"{stats['transactions']} giỏ | {stats['items']} mặt hàng phân biệt "
      f"| {stats['occurrences']} lượt mua")
print(f"độ dài giỏ: trung bình {stats['avg_len']:.4f}, nhỏ nhất {stats['min_len']}, "
      f"lớn nhất {stats['max_len']} | độ dày {stats['density']:.4f}")
print("\n5 giỏ đầu:")
for i, basket in enumerate(baskets[:5], start=1):
    print(f"  {i}. " + ", ".join(basket))

In [ ]:
# 10 mặt hàng bán chạy nhất. Con số này quan trọng cho mục 6: whole milk có mặt
# trong 1/4 số giỏ, nên mọi luật dẫn tới nó đều có confidence cao sẵn.
dem = Counter(item for basket in baskets for item in basket)
pd.DataFrame([(ten, so, round(so / len(baskets), 4)) for ten, so in dem.most_common(10)],
             columns=["mặt hàng", "số giỏ", "support"],
             index=range(1, 11))

In [ ]:
lengths = [len(basket) for basket in baskets]

fig, ax = plt.subplots()
ax.hist(lengths, bins=range(1, stats["max_len"] + 2), color=BLUE,
        edgecolor=SURFACE, linewidth=0.8)
ax.axvline(stats["avg_len"], color=ORANGE, linewidth=2, zorder=3)
ax.annotate(f"trung bình {stats['avg_len']:.2f} món",
            xy=(stats["avg_len"], 2100), xytext=(8, 0),
            textcoords="offset points", color=ORANGE, fontweight="bold", va="center")
ax.set_title("Phần lớn giỏ hàng chỉ có 1–3 món")
ax.set_xlabel("số món trong giỏ")
ax.set_ylabel("số giỏ")
ax.set_xlim(0, 33)
ax.grid(axis="x", visible=False)
plt.show()

print(f"{sum(1 for n in lengths if n <= 3) / len(lengths):.1%} số giỏ có nhiều nhất 3 món.")
print("Hệ quả: CSDL này rất THƯA (độ dày 2,6%), nên minsup phải đặt rất thấp —")
print("một luật phủ 1% số giỏ ở đây đã là luật đáng chú ý.")

## 2. Biến đổi dữ liệu thành giao dịch

Hai bước, đúng như đề yêu cầu phải nêu rõ:

1. **Một giỏ là một giao dịch.** Ở bộ này mỗi dòng đã là một giỏ nên không phải
   gộp gì. Nếu dữ liệu ở dạng mỗi dòng một món (ví dụ bộ Kaggle
   `Groceries_dataset.csv` có cột `Member_number, Date, itemDescription`) thì
   phải gộp theo cặp (khách, ngày) trước — bước đó mới là chỗ dễ sai.
2. **Mã hoá item sang số nguyên** bằng `src.utils.encode`, vì `fpgrowth` của
   nhóm làm việc trên item số (đúng định dạng SPMF). `encode` gán số theo **tên
   đã sắp a–z**, nên cùng một file dữ liệu luôn cho cùng một bảng mã — điều kiện
   để kết quả tái lập được. Đổi ngược về tên bằng `invert_code`.

Mỗi giao dịch chỉ tính một item một lần (`fpgrowth` dùng `set(transaction)` khi
đếm), nên dữ liệu có món lặp vẫn cho support đúng.

In [ ]:
encoded, code = encode(baskets)
name_of = invert_code(code)
N = len(encoded)

print(f"N = {N} giao dịch, {len(code)} mã item")
print("\nVài mã:")
for ten in list(code)[:3] + ["whole milk", "yogurt"]:
    print(f"  {code[ten]:>4}  <-  {ten}")
print("\nGiỏ đầu tiên:", baskets[0])
print("          ->  ", encoded[0])

# đổi ngược lại phải về đúng dữ liệu gốc (so theo tập, vì encode không giữ thứ tự)
assert [sorted(b) for b in baskets] == [sorted(name_of[i] for i in t) for t in encoded]
print("\nencode -> decode khớp dữ liệu gốc.")

## 3. Khai thác tập phổ biến bằng FP-Growth của nhóm

`min_count` là support **tuyệt đối**, tính bằng `to_min_count` trong
`src/config.py` (`ceil(round(minsup * N, 9))`) — đúng cách SPMF làm tròn lên, nên
kết quả so khớp được với SPMF ở Chương 1 và 2.

In [ ]:
minsup = APP1_MINSUP
min_count = to_min_count(minsup, N)

start = time.perf_counter()
itemsets = fpgrowth(encoded, min_count)
t_mine = time.perf_counter() - start

by_size = Counter(len(itemset) for itemset in itemsets)
print(f"minsup {minsup} -> min_count {min_count} (một luật phải phủ ít nhất "
      f"{min_count} giỏ)")
print(f"{len(itemsets)} tập phổ biến trong {t_mine:.3f}s\n")
print(pd.DataFrame(sorted(by_size.items()), columns=["số item", "số tập phổ biến"])
      .to_string(index=False))

sup_wm = itemsets[(code["whole milk"],)]
print(f"\nMỐC 1 — support(whole milk) = {sup_wm}/{N} = {sup_wm / N:.5f}"
      f"   (giá trị công bố: 2513, tức 0,25552)")

## 4. Sinh luật kết hợp

Với mỗi tập phổ biến $Z$ có từ 2 item trở lên, cắt $Z$ thành $X$ (bên trái) và
$Y$ (bên phải) rồi tính bốn đo lường. Gọi $N$ là số giao dịch và $sup(S)$ là số
giao dịch chứa trọn $S$:

$$
\text{support} = \frac{sup(Z)}{N},\qquad
\text{confidence} = \frac{sup(Z)}{sup(X)},\qquad
\text{lift} = \frac{\text{confidence}}{sup(Y)/N},\qquad
\text{leverage} = \frac{sup(Z)}{N} - \frac{sup(X)}{N}\cdot\frac{sup(Y)}{N}
$$

Mỗi đo lường trả lời một câu hỏi khác nhau:

| Đo lường | Câu hỏi | Đọc thế nào |
|---|---|---|
| support | Luật này phủ bao nhiêu giỏ? | Có đáng làm không |
| confidence | Mua $X$ thì bao nhiêu phần mua $Y$? | Dự đoán được không |
| lift | So với mua $Y$ một cách độc lập thì hơn bao nhiêu lần? | Có thật là liên quan không (1 = độc lập) |
| leverage | Quan hệ đó đổi ra bao nhiêu phần giỏ? | Hiệu ứng tuyệt đối |

`sup(X)` và `sup(Y)` luôn tra được trong dict kết quả: $X$ và $Y$ là tập con thực
sự của $Z$ phổ biến, nên theo tính chất đóng xuống của Apriori chúng cũng phổ
biến, và `fpgrowth` trả về **toàn bộ** tập phổ biến kể cả tập một phần tử.

Mặc định `max_consequent=1` — bên phải chỉ một món. Đây là dạng dùng được trong
bán lẻ ("khách mua X thì gợi ý thêm Y") và cũng là dạng đọc được; đặt `None` thì
xét cả $2^k - 2$ cách tách.

In [ ]:
start = time.perf_counter()
rules = generate_rules(itemsets, N, APP1_MINCONF)
t_rules = time.perf_counter() - start

print(f"minconf {APP1_MINCONF}: {len(rules)} luật, sinh trong {t_rules:.3f}s")
print(f"minconf 0.5 : {len(generate_rules(itemsets, N, 0.5))} luật")
print(f"\nMỐC 2 — minsup 0,01 với minconf 0,5 cho đúng 15 luật (giá trị công bố: 15)")


def bang_luat(rules, n=10, by="lift"):
    """Bảng n luật đứng đầu theo một tiêu chí, đã đổi số về tên món."""
    rows = [{
        "luật": format_rule(rule, name_of),
        "giỏ": rule.support_count,
        "support": round(rule.support, 4),
        "confidence": round(rule.confidence, 3),
        "lift": round(rule.lift, 3),
        "leverage": round(rule.leverage, 5),
    } for rule in sort_rules(rules, by=by)[:n]]
    return pd.DataFrame(rows, index=range(1, len(rows) + 1))


def tim_luat(rules, trai, phai):
    """Tìm đúng một luật theo tên món."""
    for rule in rules:
        left, right = rule_names(rule, name_of)
        if set(left) == set(trai) and set(right) == set(phai):
            return rule
    raise KeyError(f"không có luật {trai} -> {phai} ở ngưỡng hiện tại")


moc = tim_luat(rules, ["other vegetables", "yogurt"], ["whole milk"])
print(f"\nMỐC 3 — {format_rule(moc, name_of)}")
print(f"  nhóm tính: support {moc.support:.5f} | confidence {moc.confidence:.5f} "
      f"| lift {moc.lift:.5f}")
print(f"  công bố:   support 0,02226 | confidence 0,51280 | lift 2,00720")

## 5. Top 10 luật theo lift và ý nghĩa kinh doanh

Đây là sản phẩm chính mà đề yêu cầu.

In [ ]:
bang_luat(rules, 10, by="lift")

### Diễn giải

Mười luật này không phải mười chuyện rời rạc — chúng rơi vào **ba nhóm hành vi**,
và mỗi nhóm cho một việc làm được:

**Nhóm 1 — giỏ nấu ăn tươi (7 trong 10 luật: 1, 2, 4, 5, 6, 8, 9).**
`root vegetables`, `other vegetables`, `citrus fruit`, `tropical fruit` kéo nhau
rất mạnh (lift 2,6–3,3). Khách mua rau củ hầu như không mua đúng một loại: họ
đang mua nguyên liệu cho một bữa nấu ở nhà, nên lấy cả củ, cả rau lá, cả trái
cây trong một lượt. Đáng chú ý là luật 1 và 4 là **hai cách tách khác nhau của
cùng một tập phổ biến 3 món** `{citrus fruit, other vegetables, root vegetables}`
— cả hai cách tách đều lọt top 10, nghĩa là ba món này đi cùng nhau thành một
cụm chứ không phải "món A dẫn tới món B". Luật 2 và 5 cũng là một cặp như vậy
với `tropical fruit` thay cho `citrus fruit`.

> **Việc làm được:** xếp các quầy rau củ và trái cây liền nhau thành một khu
> "nguyên liệu nấu ăn". Nếu đang để trái cây ở đầu kia siêu thị thì đó là đang
> cắt đôi một giỏ hàng tự nhiên.

**Nhóm 2 — giỏ món hầm (luật 3).** `beef -> root vegetables`, lift 3,04. Đây là
luật **duy nhất trong top 10 có bên trái chỉ một món**, nên cũng là luật dễ dùng
nhất: chỉ cần biết khách đã lấy thịt bò là đã gợi ý được. Thịt bò và củ là công
thức món hầm kinh điển.

> **Việc làm được:** đặt kệ củ nhỏ ngay tại quầy thịt, hoặc combo "thịt bò + củ"
> kèm công thức. Hiệu ứng ước lượng được ở ô dưới.

**Nhóm 3 — giỏ bữa sáng sữa (luật 7, 10).** `curd + whole milk -> yogurt` và
`tropical fruit + whole milk -> yogurt`, lift 2,6–2,8. Người mua đồ sữa mua theo
cả nhóm, và trái cây nhiệt đới đi cùng sữa chua là mô-típ bữa sáng.

> **Việc làm được:** đưa sữa chua về cạnh quầy sữa thay vì để ở khu tráng miệng.

**Một quan sát về quy mô, cần nói thẳng:** support của cả 10 luật chỉ 1–2,3%,
tức mỗi luật phủ khoảng 100–230 giỏ trong 9.835 giỏ. Ở bán lẻ đó là chuyện bình
thường — không có luật nào phủ phần lớn hoá đơn, vì giỏ trung bình chỉ 4,4 món
trên 169 mặt hàng. Giá trị của phân tích này là **nhiều luật nhỏ cộng lại**, chứ
không phải một luật lớn. Cột `leverage` nói đúng điều đó: luật 6
(`other vegetables + whole milk -> root vegetables`) có lift thấp nhất trong top
10 nhưng leverage **cao nhất**, vì nó phủ 228 giỏ — nếu chỉ được chọn một luật để
làm thì chọn luật này, không phải luật số 1.

In [ ]:
# Ước lượng hiệu ứng của một combo, dựa trên đúng số liệu ở trên.
beef = tim_luat(rules, ["beef"], ["root vegetables"])
sup_beef = itemsets[(code["beef"],)]
sup_rv = itemsets[(code["root vegetables"],)]

ky_vong_doc_lap = sup_beef * sup_rv / N          # nếu hai món độc lập
print(f"Giỏ có beef: {sup_beef} | giỏ có root vegetables: {sup_rv}")
print(f"Giỏ có cả hai: {beef.support_count}")
print(f"Nếu hai món độc lập thì chỉ còn khoảng {ky_vong_doc_lap:.0f} giỏ.")
print(f"=> {beef.support_count - ky_vong_doc_lap:.0f} giỏ 'tăng thêm' so với ngẫu nhiên, "
      f"tức {beef.leverage:.2%} tổng số giỏ.")
print(f"\nLeverage cao nhất trong top 10 theo lift:")
print(bang_luat(rules, 10).sort_values("leverage", ascending=False)
      .head(3)[["luật", "support", "lift", "leverage"]].to_string())

## 6. Vì sao xếp theo lift chứ không phải confidence

Đề yêu cầu xếp top 10 theo lift. Lý do nằm ở chính dữ liệu này: `whole milk` có
mặt trong **1/4 số giỏ**. Bất kỳ luật nào dẫn tới `whole milk` cũng có confidence
cao sẵn, không cần hai món có quan hệ gì — đoán "khách sẽ mua sữa" thì đúng
25,6% số lần ngay cả khi chẳng biết gì về giỏ.

Trong dữ liệu này có một cặp đối chứng gần như hoàn hảo cho chuyện đó.

In [ ]:
cap = [tim_luat(rules, ["curd", "yogurt"], ["whole milk"]),
       tim_luat(rules, ["citrus fruit", "root vegetables"], ["other vegetables"])]

sup_ov = itemsets[(code["other vegetables"],)]
print(f"support(whole milk)       = {sup_wm / N:.4f}")
print(f"support(other vegetables) = {sup_ov / N:.4f}\n")

pd.DataFrame([{
    "luật": format_rule(rule, name_of),
    "support": round(rule.support, 4),
    "confidence": round(rule.confidence, 3),
    "lift": round(rule.lift, 3),
} for rule in cap], index=["dẫn tới món rất phổ biến", "dẫn tới món ít phổ biến hơn"])

In [ ]:
# Nếu xếp theo confidence thì top 5 bị món bán chạy chiếm chỗ:
bang_luat(rules, 5, by="confidence")

Hai luật ở bảng trên có confidence **gần như bằng nhau** (0,582 so với 0,586)
nhưng lift lệch tới 33%, chỉ vì `whole milk` phổ biến hơn `other vegetables`
(0,2555 so với 0,1935). Nếu xếp theo confidence, phần lớn top 5 là các luật dẫn
tới `whole milk` — tức là ta đang phát hiện lại "sữa bán chạy", một điều đã biết
từ bảng ở mục 1, chứ không phát hiện quan hệ nào mới.

Kết luận thực hành: **lift để tìm ra quan hệ, support và leverage để quyết định
có làm hay không, confidence để ước lượng tỉ lệ thành công khi gợi ý.** Bỏ một
trong bốn là ra quyết định sai — xem lại luật 6 ở mục 5.

## 7. Null model: các mẫu này có thật, hay chỉ do vài món bán chạy?

Mục 6 cho thấy độ bán chạy của từng món bóp méo confidence. Câu hỏi tiếp theo
nghiêm trọng hơn: **nếu chỉ riêng độ bán chạy của từng món đã đủ sinh ra những
luật như trên thì sao?** Khi đó cả phân tích này vô nghĩa.

Cách kiểm: dựng một CSDL đối chứng (*null model*) bằng cách **trộn các món giữa
các giỏ**, nhưng giữ **nguyên vẹn**:

- tần suất mỗi món (`whole milk` vẫn có mặt đúng 2.513 lần),
- độ dài mỗi giỏ (giỏ 3 món vẫn 3 món).

Chỉ có **việc món nào đi với món nào** bị phá. Nếu các luật ở mục 5 chỉ là hệ quả
của độ bán chạy, dữ liệu trộn phải cho ra lượng luật và mức lift tương đương.

Cách trộn: làm phẳng toàn bộ 43.367 lượt mua thành một danh sách, trộn một lần
bằng `random.Random(seed)`, rồi cắt lại theo đúng độ dài từng giỏ
(`shuffle_keep_marginals`). Seed cố định nên chạy lại ra đúng số cũ. Hạn chế đã
biết: cách này có thể đặt hai lượt của cùng một món vào một giỏ, nên dữ liệu trộn
hơi loãng hơn thật — lệch này chỉ làm null model *dễ thắng hơn*, nên kết luận
không bị thổi phồng.

In [ ]:
def do_mot_csdl(data, nhan):
    its = fpgrowth(data, min_count)
    rs = generate_rules(its, N, APP1_MINCONF)
    lifts = [rule.lift for rule in rs]
    return {"dữ liệu": nhan,
            "tập phổ biến": len(its),
            "số luật": len(rs),
            "lift lớn nhất": round(max(lifts), 3) if lifts else 0.0,
            "số luật lift > 1,5": sum(1 for lift in lifts if lift > 1.5)}


hang = [do_mot_csdl(encoded, "THẬT")]
for i in range(APP1_SHUFFLES):
    hang.append(do_mot_csdl(shuffle_keep_marginals(encoded, SEED + i),
                            f"trộn, seed {SEED + i}"))

null_df = pd.DataFrame(hang)
null_df

In [ ]:
thuc = null_df.iloc[0]
tron = null_df.iloc[1:]

print("Dữ liệu thật so với trung vị các lần trộn:")
for cot in ("số luật", "lift lớn nhất", "số luật lift > 1,5"):
    trung_vi = tron[cot].median()
    ti_so = thuc[cot] / trung_vi if trung_vi else float("inf")
    print(f"  {cot:<20} thật {thuc[cot]:>9.3f} | trộn {trung_vi:>9.3f} "
          f"| gấp {ti_so:>7.2f} lần")

Dữ liệu thật có **125 luật**, lift cao nhất **3,30**, và **108 luật** có lift trên
1,5. Năm lần trộn chỉ cho khoảng **11 luật**, lift cao nhất khoảng **1,73**, và
chừng **5 luật** vượt lift 1,5 — tức là **gấp hơn 20 lần** ở thống kê tách tốt
nhất. Vì phép trộn đã giữ nguyên độ bán chạy của từng món, phần chênh lệch này
chỉ còn một cách giải thích: **các món thật sự được khách mua cùng nhau.** Các
luật ở mục 5 không phải hiện tượng giả do vài món bán chạy.

Chú ý: dữ liệu trộn vẫn cho ra 245–253 tập phổ biến, gần bằng 333 của dữ liệu
thật. Tức là **số tập phổ biến không phải thống kê tốt để đánh giá** — nó phần
lớn do độ bán chạy của từng món quyết định. Phải nhìn vào số *luật* và vào phân
bố *lift*. Mục 9 còn cho thấy ngay cả ngưỡng "lift > 1,5" cũng mất tác dụng khi
CSDL quá thưa.

## 8. Ảnh hưởng của minsup và minconf

Hai ngưỡng này không đối xứng. Ô dưới quét 5 mức minsup × 7 mức minconf (danh
sách chốt trong `src/config.py`), mỗi mức minsup chỉ chạy `fpgrowth` một lần rồi
lọc lại nhiều minconf.

In [ ]:
quet = []
for minsup_i in APP1_MINSUPS:
    mc = to_min_count(minsup_i, N)
    its = fpgrowth(encoded, mc)
    for minconf_i in APP1_MINCONFS:
        rs = generate_rules(its, N, minconf_i)
        lifts = [rule.lift for rule in rs]
        quet.append({"minsup": minsup_i, "minconf": minconf_i, "min_count": mc,
                     "tập phổ biến": len(its), "số luật": len(rs),
                     "lift lớn nhất": max(lifts) if lifts else 0.0})

quet_df = pd.DataFrame(quet)
print("Số luật theo (minconf, minsup):")
quet_df.pivot(index="minconf", columns="minsup", values="số luật")

In [ ]:
mau = dict(zip(sorted(APP1_MINSUPS, reverse=True), RAMP))   # minsup càng thấp, màu càng đậm

fig, ax = plt.subplots()
for minsup_i, nhom in quet_df.groupby("minsup", sort=True):
    nhom = nhom.sort_values("minconf")
    ax.plot(nhom["minconf"], nhom["số luật"], color=mau[minsup_i], marker="o",
            markeredgecolor=SURFACE, markeredgewidth=1.2, label=f"minsup {minsup_i}")

ax.set_yscale("symlog", linthresh=1)
ax.set_title("minconf siết số luật mạnh hơn nhiều so với minsup")
ax.set_xlabel("minconf")
ax.set_ylabel("số luật (thang symlog)")
ax.legend(title="", loc="upper right", labelcolor=INK_2)
plt.show()

In [ ]:
mot_minconf = quet_df[quet_df["minconf"] == APP1_MINCONF].sort_values("minsup")

fig, ax = plt.subplots()
for cot, color, marker in (("tập phổ biến", BLUE, "o"), ("số luật", ORANGE, "s")):
    ax.plot(mot_minconf["minsup"], mot_minconf[cot], color=color, marker=marker,
            markeredgecolor=SURFACE, markeredgewidth=1.2)
    # nhãn đặt ở đầu minsup CAO, nơi hai đường còn tách xa nhau
    dau = mot_minconf.iloc[-1]
    ax.annotate(cot, xy=(dau["minsup"], dau[cot]), xytext=(-8, 0),
                textcoords="offset points", color=color, fontweight="bold",
                ha="right", va="center")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.055, 0.0008)       # giới hạn giảm dần = trục đảo chiều
ax.set_title(f"Hạ minsup: số luật bùng nổ nhanh hơn số tập phổ biến (minconf {APP1_MINCONF})")
ax.set_xlabel("minsup (giảm dần sang phải, thang log)")
ax.set_ylabel("số lượng (thang log)")
plt.show()

cao, thap = mot_minconf.iloc[-1], mot_minconf.iloc[0]
print(f"Từ minsup {cao['minsup']} xuống {thap['minsup']}:")
print(f"  tập phổ biến {cao['tập phổ biến']:>6} -> {thap['tập phổ biến']:>6} "
      f"(gấp {thap['tập phổ biến'] / cao['tập phổ biến']:.0f} lần)")
print(f"  số luật      {cao['số luật']:>6} -> {thap['số luật']:>6} "
      f"(gấp {thap['số luật'] / cao['số luật']:.0f} lần)")

Hai hình trên nói ba điều:

1. **minconf siết mạnh hơn minsup.** Ở minsup 0,02, nâng minconf từ 0,2 lên 0,5
   là từ 72 luật còn 1 luật, và từ 0,6 trở lên là không còn luật nào — vì giỏ
   trung bình chỉ 4,4 món nên gần như không có món nào "kéo theo" món khác với
   xác suất trên 60%.
2. **Số luật bùng nổ nhanh hơn số tập phổ biến.** Hạ minsup từ 0,02 xuống 0,001
   làm số tập phổ biến gấp khoảng 110 lần, nhưng số luật gấp khoảng 370 lần. Lý
   do nằm ở cấu trúc: một tập phổ biến $k$ item sinh ra $k$ luật khi bên phải chỉ
   một món, mà hạ minsup thì các tập *dài* mới là phần tăng nhanh nhất (ở minsup
   0,001 đã có tập 6 item). Đây chính là mặt "phía luật" của hiện tượng bùng nổ
   tổ hợp mà Chương 2 đo ở phía tập phổ biến.
3. **Lift cao nhất tăng theo khi hạ minsup** — 2,8 ở minsup 0,02 lên 35,7 ở
   0,001. Đừng đọc đó là "tìm được quan hệ mạnh hơn": đó là các cặp món ngách,
   mẫu số $sup(Y)/N$ rất nhỏ nên lift vọt lên, mà cả luật chỉ dựa trên 10 giỏ.
   Mục 9 cho thấy chuyện này nghiêm trọng tới mức nào trên CSDL thưa hơn.

In [ ]:
# Hình kinh điển của phân tích giỏ hàng: support × confidence, tô màu theo lift.
r_thap = generate_rules(fpgrowth(encoded, to_min_count(0.005, N)), N, 0.2)

fig, ax = plt.subplots()
diem = ax.scatter([rule.support for rule in r_thap],
                  [rule.confidence for rule in r_thap],
                  c=[rule.lift for rule in r_thap], cmap=SEQ,
                  s=30, edgecolor=SURFACE, linewidth=0.5)
thanh = fig.colorbar(diem, ax=ax)
thanh.set_label("lift", color=INK_2, rotation=270, labelpad=14)
thanh.outline.set_visible(False)

ax.set_title("Lift cao dồn về góc support thấp")
ax.set_xlabel("support")
ax.set_ylabel("confidence")
plt.show()

print(f"{len(r_thap)} luật ở minsup 0,005 và minconf 0,2.")
print("Các điểm đậm nhất (lift cao) dồn hết về mép trái: lift cao gần như luôn đi")
print("với support thấp, nên không bao giờ đọc lift một mình.")

## 9. Chạy trên CSDL lớn: Retail, và bài học về lift ở dữ liệu thưa

`data/benchmark/retail.txt` — 88.162 giỏ của một nhà bán lẻ Bỉ, 16.470 mặt hàng,
cũng là một trong bốn CSDL benchmark của Chương 2. Item đã bị ẩn danh thành số
nên **không giải thích được ý nghĩa kinh doanh**, nhưng chạy nó trả lời hai câu
hỏi khác: quy trình có làm việc được ở quy mô thật không, và các đo lường cư xử
thế nào khi dữ liệu thưa hơn nữa.

In [ ]:
retail_tx, _, _ = load_dataset("retail")
retail_stats = describe(retail_tx)
retail_minsup = APP1_MINSUP_RETAIL
retail_min_count = to_min_count(retail_minsup, len(retail_tx))

start = time.perf_counter()
retail_itemsets = fpgrowth(retail_tx, retail_min_count)
t_retail = time.perf_counter() - start
retail_rules = generate_rules(retail_itemsets, len(retail_tx), APP1_MINCONF)

print(f"minsup {retail_minsup} -> min_count {retail_min_count}")
print(f"{len(retail_itemsets)} tập phổ biến trong {t_retail:.2f}s, "
      f"{len(retail_rules)} luật")

pd.DataFrame([
    {"CSDL": "Groceries", "giao dịch": stats["transactions"],
     "mặt hàng": stats["items"], "giỏ trung bình": round(stats["avg_len"], 2),
     "độ dày": round(stats["density"], 5), "minsup": minsup,
     "min_count": min_count, "tập phổ biến": len(itemsets),
     "số luật": len(rules),
     "lift lớn nhất": round(max(r.lift for r in rules), 1)},
    {"CSDL": "Retail", "giao dịch": retail_stats["transactions"],
     "mặt hàng": retail_stats["items"], "giỏ trung bình": round(retail_stats["avg_len"], 2),
     "độ dày": round(retail_stats["density"], 5), "minsup": retail_minsup,
     "min_count": retail_min_count, "tập phổ biến": len(retail_itemsets),
     "số luật": len(retail_rules),
     "lift lớn nhất": round(max(r.lift for r in retail_rules), 1)},
]).set_index("CSDL")

In [ ]:
# Top 5 luật của Retail. Item là số nên chỉ đọc được về mặt định lượng.
pd.DataFrame([{
    "luật": format_rule(rule),
    "giỏ": rule.support_count,
    "support": round(rule.support, 5),
    "confidence": round(rule.confidence, 3),
    "lift": round(rule.lift, 1),
} for rule in sort_rules(retail_rules)[:5]], index=range(1, 6))

In [ ]:
# Null model trên Retail: kết luận KHÁC hẳn Groceries.
def do_retail(data, nhan):
    its = fpgrowth(data, retail_min_count)
    rs = generate_rules(its, len(retail_tx), APP1_MINCONF)
    lifts = [rule.lift for rule in rs]
    return {"dữ liệu": nhan, "tập phổ biến": len(its), "số luật": len(rs),
            "lift lớn nhất": round(max(lifts), 2) if lifts else 0.0,
            "số luật lift > 1,5": sum(1 for lift in lifts if lift > 1.5)}


hang_retail = [do_retail(retail_tx, "THẬT")]
for i in range(3):          # 3 lần trộn là đủ để thấy, Retail chạy chậm hơn
    hang_retail.append(do_retail(shuffle_keep_marginals(retail_tx, SEED + i),
                                 f"trộn, seed {SEED + i}"))

pd.DataFrame(hang_retail)

### Hai bài học, và một chỗ phải thú nhận

**Bài học 1 — lift không ổn định khi support thấp.** Retail cho lift cao nhất tới
**338**, gấp hơn 100 lần mức 3,3 của Groceries. Đó **không** phải vì quan hệ mạnh
hơn, mà vì mẫu số $sup(Y)/N$ rất nhỏ: cả luật chỉ dựa trên 91 giỏ trong 88.162
giỏ. Ước lượng dựa trên 91 quan sát có phương sai lớn, và hạ minsup thêm thì lift
cao nhất lại tăng. Kết luận phương pháp: **ở CSDL thưa, không xếp hạng bằng lift
trần.** Phải kèm ngưỡng support tuyệt đối tối thiểu, hoặc chuyển sang leverage —
đo lường này tự động phạt các luật support thấp.

**Bài học 2 — null model phải đọc theo đúng thống kê.** Trên Groceries, dữ liệu
trộn sụp ở cả ba thống kê. Trên Retail thì **không**: số luật chỉ giảm khoảng 1,5
lần, và số luật có lift > 1,5 của dữ liệu **trộn còn nhiều hơn** dữ liệu thật.
Chỉ phần đỉnh của phân bố lift mới tách ra rõ (338 so với khoảng 2,7). Lý do:
Retail có độ dày 0,0006 — khi support nền của mỗi món đã cực bé thì hai món ngẫu
nhiên rơi vào cùng giỏ cũng đủ cho lift vượt 1,5. Nghĩa là **ngưỡng "lift > 1,5"
không chứng minh được gì trên CSDL thưa**; đó là một kết quả âm, và nó đáng báo
cáo đúng như nó là.

**Chỗ phải thú nhận:** vì item Retail đã ẩn danh, không ai kiểm được luật
`{697} -> {700}` có hợp lý hay không. Số liệu vẫn đúng, nhưng giá trị sử dụng
bằng không. Bài học thực tế: **giá trị của luật kết hợp phụ thuộc vào việc có từ
điển item hay không, chứ không phụ thuộc thuật toán.** Khi triển khai thật, việc
giữ được bảng mã món là phần quan trọng ngang với việc chạy được thuật toán.

## 10. Kết luận

**Đã làm được gì.** Toàn bộ quy trình phân tích giỏ hàng chạy trên cài đặt
FP-Growth của nhóm: đọc dữ liệu thật, biến đổi thành giao dịch, khai thác tập phổ
biến, sinh luật kết hợp với bốn đo lường, và diễn giải. Trên Groceries (9.835
giỏ, 169 mặt hàng) ở minsup 0,01 và minconf 0,3 ra 333 tập phổ biến và 125 luật
trong dưới 0,1 giây. Quy trình chạy lại nguyên vẹn trên Retail (88.162 giỏ,
16.470 mặt hàng) trong khoảng 3 giây.

**Bốn mốc kiểm chứng** — đối chiếu **kết quả số** với các giá trị đã công bố của
bộ Groceries, không đọc và không sao chép code của bất kỳ cài đặt nào:

| Mốc | Nhóm tính được | Giá trị công bố |
|---|---|---|
| support(`whole milk`) | 2513 / 9835 = 0,25552 | 0,25552 |
| `{other vegetables, yogurt} -> {whole milk}` | sup 0,02227 · conf 0,51288 · lift 2,00723 | 0,02226 · 0,5128 · 2,0072 |
| số luật tại minsup 0,01 và minconf 0,5 | 15 | 15 |
| số luật tại minsup 0,001 và minconf 0,8 | 410 | 410 |

Khớp ở cả hai đầu của dải tham số (333 tập phổ biến và 13.492 tập phổ biến) thì
khó mà đúng do may.

**Ba đề xuất hành động cho cửa hàng**, theo thứ tự hiệu ứng tuyệt đối
(`leverage`), không theo lift:

1. Gộp quầy rau củ và trái cây thành một khu "nguyên liệu nấu ăn" — nhóm luật này
   chiếm 7 trong 10 luật mạnh nhất, và luật
   `{other vegetables, whole milk} -> {root vegetables}` một mình đã phủ 228 giỏ.
2. Đặt kệ củ tại quầy thịt bò — luật `{beef} -> {root vegetables}` là luật dễ
   dùng nhất vì chỉ cần một món bên trái.
3. Chuyển sữa chua về cạnh quầy sữa, bỏ khỏi khu tráng miệng.

**Bốn kết luận về phương pháp**, là phần nhóm thấy đáng giá nhất:

1. **Bốn đo lường, bốn câu hỏi khác nhau, không thay thế được nhau.** Xếp theo
   confidence thì top 5 bị `whole milk` chiếm chỗ và ta chỉ phát hiện lại "sữa
   bán chạy". Xếp theo lift trần thì chọn phải các cặp ngách chỉ dựa trên vài
   chục giỏ. `leverage` là đo lường ra quyết định được, vì nó phạt support thấp.
2. **Null model là bước nên có.** Trộn món giữa các giỏ nhưng giữ nguyên độ bán
   chạy cho thấy dữ liệu thật sinh ra gấp hơn 20 lần số luật lift mạnh — bằng
   chứng các mẫu là thật. Chi phí gần như bằng không vì dùng lại chính `fpgrowth`.
3. **Nhưng kết luận của null model phụ thuộc độ thưa của dữ liệu.** Trên Retail,
   ngưỡng "lift > 1,5" mất hoàn toàn tác dụng — dữ liệu trộn còn vượt ngưỡng
   nhiều hơn dữ liệu thật. Phải chọn thống kê so sánh theo từng CSDL, và đây là
   lý do `compare_real_vs_shuffled` trong `scripts/app1_curves.py` cố ý in cả ba
   tỉ số thay vì viết sẵn một câu kết luận.
4. **Số luật bùng nổ nhanh hơn số tập phổ biến** (370 lần so với 110 lần khi hạ
   minsup 20 lần). Nối tiếp phần bùng nổ tổ hợp mà Chương 2 đo ở phía tập phổ
   biến, và là lý do phải lọc luật chứ không chỉ lọc tập phổ biến.

**Hạn chế, nói thẳng để trả lời vấn đáp.** (a) `max_consequent=1` nên không sinh
luật có nhiều món bên phải; trên Groceries ở minsup 0,01 việc này chỉ bỏ sót 3
luật (231 so với 234 ở minconf 0,2), nhưng ở minsup 0,001 thì bỏ sót khoảng 14%
(21.633 so với 25.022). (b) Null model có thể đặt hai lượt của cùng một món vào một giỏ, nên dữ liệu
trộn hơi loãng hơn thật; lệch này làm null model dễ thắng hơn nên không thổi
phồng kết luận. (c) Chưa kiểm định thống kê theo nghĩa p-value, chỉ so với trung
vị của 5 lần trộn. (d) Dữ liệu Groceries không có giá và số lượng, nên không ước
lượng được doanh thu của một combo, chỉ ước lượng được số giỏ.

**Tái lập.** Mọi ngưỡng nằm trong `src/config.py`, `SEED = 42` ở mọi chỗ có ngẫu
nhiên. Chạy lại ngoài notebook:

```bash
python -m scripts.app1_basket --minsup 0.01 --minconf 0.3 --top 10
python -m scripts.app1_basket --minsup 0.01 --minconf 0.5     # phải ra đúng 15 luật
python -m scripts.app1_curves --dataset groceries
python -m scripts.app1_curves --dataset retail
python -m pytest tests/test_rules.py tests/test_basket.py -v
```

## Tài liệu tham khảo

1. J. Han, J. Pei, Y. Yin, R. Mao. *Mining Frequent Patterns without Candidate
   Generation: A Frequent-Pattern Tree Approach.* Data Mining and Knowledge
   Discovery, 8(1):53–87, 2004. — thuật toán FP-Growth mà nhóm cài đặt lại.
2. R. Agrawal, R. Srikant. *Fast Algorithms for Mining Association Rules.*
   VLDB 1994. — định nghĩa luật kết hợp, support và confidence.
3. S. Brin, R. Motwani, J. D. Ullman, S. Tsur. *Dynamic Itemset Counting and
   Implication Rules for Market Basket Data.* SIGMOD 1997. — lift (interest).
4. M. Hahsler, B. Grün, K. Hornik. *arules — A Computational Environment for
   Mining Association Rules and Frequent Item Sets.* Journal of Statistical
   Software, 14(15), 2005. — **nguồn dữ liệu Groceries và các giá trị số dùng để
   kiểm chứng**, không phải nguồn mã nguồn.
5. P.-N. Tan, M. Steinbach, V. Kumar. *Introduction to Data Mining.* — chương về
   interestingness measures (lift, leverage, conviction) và về mô hình đối chứng.